In [3]:
import numpy as np
from numba import njit

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

import scipy.linalg as la

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.metrics import root_mean_squared_error, r2_score

import optuna

import warnings

# Init

In [4]:
steps = 20000

tau_steps = 1

transient_steps_henon = int(steps * 0.1)
transient_steps_reservoir = int(steps * 0.1)

total_steps = steps + transient_steps_henon + transient_steps_reservoir + tau_steps
total_steps_after_henon = steps + transient_steps_reservoir + tau_steps

test_size = 0.2
test_steps = int(steps * test_size)

t = np.arange(0, total_steps)

In [5]:
henon_dataset = np.zeros((total_steps, 2))

rng = np.random.default_rng(42)
henon_dataset[0] = rng.random(2)

a = 1.4
b = 0.3

In [6]:
@njit
def henon_numba(steps, a=1.4, b=0.3, x0=0.0, y0=0.0):
    X = np.zeros(steps)
    Y = np.zeros(steps)
    X[0] = x0
    Y[0] = y0

    for i in range(1, steps):
        X[i] = 1 - a * X[i - 1] ** 2 + Y[i - 1]
        Y[i] = b * X[i - 1]

    return X, Y

In [7]:
henon_data_x, henon_data_y = henon_numba(total_steps)

henon_dataset = np.column_stack((henon_data_x, henon_data_y))
henon_dataset = henon_dataset[transient_steps_henon:]

In [8]:
henon_scaler = StandardScaler()
henon_train_scaled = henon_scaler.fit_transform(henon_dataset[:-4000])
henon_test_scaled = henon_scaler.transform(henon_dataset[-4000:])
henon_scaled = np.concatenate((henon_train_scaled, henon_test_scaled), axis=0)

In [9]:
def henon_plot(data_list, names=None):
    fig = go.Figure()
    colors = ["white", "magenta"]

    for i, data in enumerate(data_list):
        fig.add_trace(
            go.Scattergl(
                x=data[:, 0],
                y=data[:, 1],
                mode="markers",
                name=names[i] if names else f"Dataset {i+1}",
                marker=dict(color=colors[i % len(colors)], size=1),
            )
        )

    fig.update_layout(
        plot_bgcolor="black",
        paper_bgcolor="black",
        font=dict(color="white"),
        xaxis=dict(
            showgrid=False,
            zeroline=False,
            linecolor="white",
            ticks="outside",
            tickcolor="white",
        ),
        yaxis=dict(
            showgrid=False,
            zeroline=False,
            linecolor="white",
            ticks="outside",
            tickcolor="white",
        ),
    )

    return fig

In [10]:
def r_2_plots_grid(actual_list, predicted_list, titles):
    fig = make_subplots(rows=1, cols=actual_list.shape[1], subplot_titles=titles)

    for i in range(actual_list.shape[1]):
        actual = actual_list[:, i]
        predicted = predicted_list[:, i]
        r_2 = r2_score(actual, predicted)
        col = i + 1

        fig.add_trace(go.Scatter(
            x=actual, y=predicted, mode="markers",
            name="Data", marker=dict(color="rgba(50, 50, 200, 0.5)", size=5)
        ), row=1, col=col)

        min_val, max_val = min(actual.min(), predicted.min()), max(actual.max(), predicted.max())
        fig.add_trace(go.Scatter(
            x=[min_val, max_val], y=[min_val, max_val], mode="lines", 
            name="Ideal", line=dict(color="firebrick", dash="dash")
        ), row=1, col=col)

        fig.update_xaxes(title_text="Actual", row=1, col=col)
        fig.update_yaxes(title_text=f"Predicted (R²: {r_2:.4f})", row=1, col=col)

    fig.update_layout(showlegend=False, height=500, width=1000)
    return fig

In [11]:
def generate_henon_grid(all_experiments, cols=3, plot_height=400):
    total_plots = len(all_experiments)
    rows = (total_plots + cols - 1) // cols
    colors = ["white", "magenta"]

    # 1. Initialize the master subplot matrix
    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=[f"System #{i+1}" for i in range(total_plots)],
        horizontal_spacing=0.04,
        vertical_spacing=0.03,
    )

    for idx, data_list in enumerate(all_experiments):
        current_row = (idx // cols) + 1
        current_col = (idx % cols) + 1

        for i, data in enumerate(data_list):
            fig.add_trace(
                go.Scattergl(
                    x=data[:, 0],
                    y=data[:, 1],
                    mode="markers",
                    marker=dict(color=colors[i % len(colors)], size=1),
                    showlegend=False,
                ),
                row=current_row,
                col=current_col,
            )

    fig.update_layout(
        height=plot_height * rows,
        plot_bgcolor="black",
        paper_bgcolor="black",
        font=dict(color="white", size=10),
        margin=dict(t=80, b=40, l=40, r=40),
    )

    fig.update_xaxes(
        showgrid=False,
        zeroline=False,
        linecolor="white",
        ticks="outside",
        tickcolor="white",
    )
    fig.update_yaxes(
        showgrid=False,
        zeroline=False,
        linecolor="white",
        ticks="outside",
        tickcolor="white",
    )

    return fig

# Bayesian Init

In [12]:
@njit(fastmath=True, cache=True)
def compute_states(steps, henon_inputs, res_size, W_in, W_res, bias, alpha, noise):
    X_data = np.zeros((steps, res_size))
    X_data[0] = 0.0

    state = np.zeros(res_size)
    input_projections = (henon_inputs + noise) @ W_in.T

    for i in range(1, steps):
        state = (1.0 - alpha) * state + alpha * np.tanh(
            input_projections[i - 1] + W_res @ state + bias
        )
        X_data[i] = state
    return X_data

In [13]:
@njit(fastmath=True, cache=True)
def compute_closed_states(
    steps_start,
    steps_end,
    Y_pred_scaled,
    X_pred,
    W_in,
    W_res,
    bias,
    alpha,
    W_out,
    W_bias,
    res_size,
    scaler_mean,
    scaler_std,
):
    for i in range(steps_start, steps_end):
        u = Y_pred_scaled[i - 2]
        prev_state = X_pred[i - 1, :res_size]
        new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
        X_pred[i, :res_size] = (1.0 - alpha) * prev_state + alpha * new_state
        X_scaled = (X_pred[i, :res_size] - scaler_mean) / scaler_std
        Y_pred_scaled[i] = W_out @ X_scaled + W_bias
    return Y_pred_scaled

In [14]:
@njit(fastmath=True, cache=True)
def lyapunov_loss(Y_test, Y_pred, a, b, total_steps_after_henon):
    Q_test = np.ascontiguousarray(np.eye(2))
    lyapunov_sums_test = np.zeros(2)
    for i in range(len(Y_test)):
        J = np.array([[-2.0 * a * Y_test[i, 0], 1.0], [b, 0.0]])
        Z = np.ascontiguousarray(J @ Q_test)
        Q_test_raw, R = np.linalg.qr(Z)
        Q_test = np.ascontiguousarray(Q_test_raw)
        lyapunov_sums_test += np.log(np.abs(np.diag(R)))
    lyapunov_exponent_test = lyapunov_sums_test / total_steps_after_henon

    Q_pred = np.ascontiguousarray(np.eye(2))
    lyapunov_sums_pred = np.zeros(2)
    for i in range(len(Y_pred)):
        J = np.array([[-2.0 * a * Y_pred[i, 0], 1.0], [b, 0.0]])
        Z = np.ascontiguousarray(J @ Q_pred)
        Q_pred_raw, R = np.linalg.qr(Z)
        Q_pred = np.ascontiguousarray(Q_pred_raw)
        lyapunov_sums_pred += np.log(np.abs(np.diag(R)))
    lyapunov_exponent_pred = lyapunov_sums_pred / total_steps_after_henon

    le_loss = (lyapunov_exponent_test - lyapunov_exponent_pred) ** 2
    return le_loss[0]

In [15]:
def normalize_2d(data):
    min_vals = data.min(axis=0)
    max_vals = data.max(axis=0)
    return (data - min_vals) / (max_vals - min_vals)

In [16]:
@njit(fastmath=True, cache=True)
def random_sparse_orthogonal(dim, density, rng):
    Q = np.eye(dim)
    target_nnz = int(dim**2 * density)
    check_frequency = max(1, dim // 40)
    for _ in range(dim**2):
        i = rng.integers(0, dim)
        j = rng.integers(0, dim)
        while i == j:
            j = rng.integers(0, dim)

        theta = rng.uniform(0, 2 * np.pi)
        c, s = np.cos(theta), np.sin(theta)

        for k in range(dim):
            temp_i = Q[i, k]
            temp_j = Q[j, k]
            Q[i, k] = c * temp_i + s * temp_j
            Q[j, k] = -s * temp_i + c * temp_j

        if _ % check_frequency == 0:
            if np.count_nonzero(Q) >= target_nnz:
                break
    return Q


N = 200
rng = np.random.default_rng(42)
sparse_orthogonal = random_sparse_orthogonal(dim=N, density=0.1, rng=rng)

print("Sample slice of random float values:")
print(sparse_orthogonal[:5, :5].round(2))

print(f"\nTotal non-zero elements: {np.count_nonzero(sparse_orthogonal)}")
print(f"Actual matrix density: {np.count_nonzero(sparse_orthogonal) / (N**2):.4f}")

product = sparse_orthogonal @ sparse_orthogonal.T
identity = np.eye(sparse_orthogonal.shape[0])
error_matrix = product - identity
orthogonality_error = np.linalg.norm(error_matrix, ord="fro")
print(f"Orthogonality Error: {orthogonality_error:.2e}")

Sample slice of random float values:
[[ 0.95  0.    0.    0.    0.  ]
 [-0.    0.01 -0.   -0.   -0.  ]
 [ 0.    0.   -0.11  0.    0.  ]
 [ 0.    0.    0.   -0.15  0.  ]
 [ 0.    0.    0.    0.    0.08]]

Total non-zero elements: 4041
Actual matrix density: 0.1010
Orthogonality Error: 2.77e-15


# Auto Deep Reservoir

In [17]:
def deep_reservoir(configs, in_size, steps, transient_steps, henon_data):
    rng = np.random.default_rng(42)
    layers = []
    current_input = henon_data

    for config in configs:
        res_size = config["res_size"]
        sparsity = config["sparsity"]
        alpha = config["alpha"]
        spec_rad = config["spec_rad"]
        input_scaling = config["input_scaling"]
        bias_scaling = config["bias_scaling"]
        noise_val = config["noise_val"]

        bias = rng.uniform(-bias_scaling, bias_scaling, size=res_size)
        W_in = rng.uniform(-input_scaling, input_scaling, (res_size, in_size))
        W_res = rng.uniform(-1.0, 1.0, size=(res_size, res_size))
        mask = rng.random((res_size, res_size)) < sparsity
        W_res *= mask
        try:
            eigenvalues = la.eigvals(W_res)
            largest_eigenvalue = np.round(np.max(np.abs(eigenvalues)), decimals=12)
            W_res = W_res * (spec_rad / largest_eigenvalue)
        except Exception as e:
            print("Eigenval Issue")
        noise = rng.normal(0, noise_val, size=(steps + transient_steps_reservoir, in_size))

        states = compute_states(
            steps + transient_steps_reservoir + tau_steps,
            current_input[:-tau_steps],
            res_size,
            W_in,
            W_res,
            bias,
            alpha,
            noise,
        )

        layers.append({"states": states, "W_in": W_in, "W_res": W_res, "bias": bias})
        current_input = states
        in_size = config["res_size"]

    return layers

In [24]:
layer_configs = [
    {
        "res_size": 200,
        "sparsity": 0.2,
        "alpha": 0.65,
        "spec_rad": 1.0,
        "bias_scaling": 0.1,
        "input_scaling": 0.4,
        "ridge_alpha": 1e-5,
        "noise_val": 0.0,
    },
    {
        "res_size": 300,
        "sparsity": 0.15,
        "alpha": 0.7,
        "spec_rad": 0.95,
        "input_scaling": 0.2,
        "bias_scaling": 0.2,
        "ridge_alpha": 1e-4,
        "noise_val": 0.0,
    },
    {
        "res_size": 200,
        "sparsity": 0.2,
        "alpha": 0.7,
        "spec_rad": 1.05,
        "bias_scaling": 0.07,
        "input_scaling": 0.2,
        "ridge_alpha": 1e-3,
        "noise_val": 0.0,
    },
]
layers = deep_reservoir(layer_configs, 2, steps, transient_steps_reservoir, henon_scaled);

In [25]:
Y_data = henon_scaled[transient_steps_reservoir + tau_steps :]
Y_train_scaled, Y_test_scaled = (
    Y_data[:-test_steps],
    Y_data[-test_steps:],
)

## Layer 1

In [26]:
X_data = layers[0]["states"][transient_steps_reservoir:-tau_steps]

x_scaler = StandardScaler()
X_train_scaled, X_test_scaled = (
    x_scaler.fit_transform(X_data[:-test_steps]),
    x_scaler.transform(X_data[-test_steps:]),
)

In [27]:
try:
    model = Ridge(alpha=layer_configs[0]["ridge_alpha"], solver="auto")
    model.fit(X_train_scaled, Y_train_scaled)
except Exception as e:
    try:
        model = Ridge(alpha=layer_configs[0]["ridge_alpha"], solver="svd")
        model.fit(X_train_scaled, Y_train_scaled)
        print("Used SVD solver")
    except Exception as e:
        print(f"Error fitting model: {e}")

In [28]:
Y_pred_scaled = model.predict(X_test_scaled)

Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
Y_test = henon_scaler.inverse_transform(Y_test_scaled)

In [29]:
rmse = root_mean_squared_error(Y_test, Y_pred)
r_2 = r2_score(Y_test, Y_pred)
rmse, r_2

(0.07782767664449432, 0.9778026063390521)

In [30]:
henon_plot([Y_test, Y_pred], names=["Actual", "Predicted"])

## Layer 2

In [31]:
X_data = layers[1]["states"][transient_steps_reservoir:-tau_steps]

x_scaler = StandardScaler()
X_train_scaled, X_test_scaled = (
    x_scaler.fit_transform(X_data[:-test_steps]),
    x_scaler.transform(X_data[-test_steps:]),
)

In [32]:
try:
    model = Ridge(alpha=layer_configs[1]["ridge_alpha"], solver="auto")
    model.fit(X_train_scaled, Y_train_scaled)
except Exception as e:
    try:
        model = Ridge(alpha=layer_configs[1]["ridge_alpha"], solver="svd")
        model.fit(X_train_scaled, Y_train_scaled)
        print("Used SVD solver")
    except Exception as e:
        print(f"Error fitting model: {e}")

In [33]:
Y_pred_scaled = model.predict(X_test_scaled)

Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
Y_test = henon_scaler.inverse_transform(Y_test_scaled)

In [34]:
rmse = root_mean_squared_error(Y_test, Y_pred)
r_2 = r2_score(Y_test, Y_pred)
rmse, r_2

(0.2906932693822111, 0.6929063743225099)

In [35]:
henon_plot([Y_test, Y_pred], names=["Actual", "Predicted"])

## Layer 3

In [36]:
X_data = layers[2]["states"][transient_steps_reservoir:-tau_steps]

x_scaler = StandardScaler()
X_train_scaled, X_test_scaled = (
    x_scaler.fit_transform(X_data[:-test_steps]),
    x_scaler.transform(X_data[-test_steps:]),
)

In [37]:
try:
    model = Ridge(alpha=layer_configs[2]["ridge_alpha"], solver="auto")
    model.fit(X_train_scaled, Y_train_scaled)
except Exception as e:
    try:
        model = Ridge(alpha=layer_configs[2]["ridge_alpha"], solver="svd")
        model.fit(X_train_scaled, Y_train_scaled)
        print("Used SVD solver")
    except Exception as e:
        print(f"Error fitting model: {e}")

In [38]:
Y_pred_scaled = model.predict(X_test_scaled)

Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
Y_test = henon_scaler.inverse_transform(Y_test_scaled)

In [39]:
rmse = root_mean_squared_error(Y_test, Y_pred)
r_2 = r2_score(Y_test, Y_pred)
rmse, r_2

(0.39570858727064806, 0.3114007073139024)

In [40]:
henon_plot([Y_test, Y_pred], names=["Actual", "Predicted"])

# Third Layer Bayesian

In [18]:
def deep_henon(
    layer_configs, steps, transient_steps_reservoir, henon_scaled, layer_index
):
    layers = deep_reservoir(layer_configs, 2, steps, transient_steps_reservoir, henon_scaled)

    X_data = layers[layer_index]["states"][transient_steps_reservoir:-tau_steps]
    x_scaler = StandardScaler()
    X_train_scaled, X_test_scaled = (
        x_scaler.fit_transform(X_data[:-test_steps]),
        x_scaler.transform(X_data[-test_steps:]),
    )

    with warnings.catch_warnings():
        warnings.filterwarnings("error", category=Warning)
        try:
            model = Ridge(
                alpha=layer_configs[layer_index]["ridge_alpha"], solver="auto"
            )
            model.fit(X_train_scaled, Y_train_scaled)
        except (Warning, ValueError):
            try:
                model = Ridge(
                    alpha=layer_configs[layer_index]["ridge_alpha"], solver="svd"
                )
                model.fit(X_train_scaled, Y_train_scaled)
            except Exception as e:
                print("Ridge nor SVD worked")
                raise optuna.exceptions.TrialPruned()

    Y_pred_scaled = model.predict(X_test_scaled)
    Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
    Y_test = henon_scaler.inverse_transform(Y_test_scaled)
    return Y_pred, Y_test

In [ ]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    layer_configs = [
        {
            "res_size": trial.suggest_int("res_size_1", 50, 300),
            "sparsity": trial.suggest_float("sparsity_1", 0.01, 0.5),
            "alpha": trial.suggest_float("alpha_1", 0.1, 0.8),
            "spec_rad": trial.suggest_float("spec_rad_1", 0.2, 2.0),
            "bias_scaling": trial.suggest_float("bias_scaling_1", 1e-2, 2.0),
            "input_scaling": trial.suggest_float("input_scaling_1", 1e-2, 2.0),
            "ridge_alpha": trial.suggest_float("ridge_alpha_1", 1e-13, 1e-3, log=True),
            "noise_val": 0.0,
        },
        {
            "res_size": trial.suggest_int("res_size_2", 50, 300),
            "sparsity": trial.suggest_float("sparsity_2", 0.01, 0.5),
            "alpha": trial.suggest_float("alpha_2", 0.1, 0.8),
            "spec_rad": trial.suggest_float("spec_rad_2", 0.2, 2.0),
            "bias_scaling": trial.suggest_float("bias_scaling_2", 1e-2, 2.0),
            "input_scaling": trial.suggest_float("input_scaling_2", 1e-2, 2.0),
            "ridge_alpha": trial.suggest_float("ridge_alpha_2", 1e-13, 1e-3, log=True),
            "noise_val": 0.0,
        },
        {
            "res_size": trial.suggest_int("res_size_3", 50, 300),
            "sparsity": trial.suggest_float("sparsity_3", 0.01, 0.5),
            "alpha": trial.suggest_float("alpha_3", 0.1, 0.8),
            "spec_rad": trial.suggest_float("spec_rad_3", 0.2, 2.0),
            "bias_scaling": trial.suggest_float("bias_scaling_3", 1e-2, 2.0),
            "input_scaling": trial.suggest_float("input_scaling_3", 1e-2, 2.0),
            "ridge_alpha": trial.suggest_float("ridge_alpha_3", 1e-13, 1e-3, log=True),
            "noise_val": 0.0,
        },
    ]

    layer_index = 2
    Y_pred, Y_test = deep_henon(
        layer_configs, steps, transient_steps_reservoir, henon_scaled, layer_index
    )
    rmse = root_mean_squared_error(Y_test, Y_pred)

    return rmse


study = optuna.create_study(directions=["minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=100, n_jobs=-1)

[Optuna] Processing Trial #99...

In [75]:
data = []
for i, trial in enumerate(study.best_trials):
    print(f"\r Processing Trial {i}/{len(study.best_trials)}...", end="", flush=True)
    params = trial.params
    layer_configs = [
        {
            "res_size": params["res_size_1"],
            "sparsity": params["sparsity_1"],
            "alpha": params["alpha_1"],
            "spec_rad": params["spec_rad_1"],
            "bias_scaling": params["bias_scaling_1"],
            "input_scaling": params["input_scaling_1"],
            "ridge_alpha": params["ridge_alpha_1"],
            "noise_val": 0,
        },
        {
            "res_size": params["res_size_2"],
            "sparsity": params["sparsity_2"],
            "alpha": params["alpha_2"],
            "spec_rad": params["spec_rad_2"],
            "bias_scaling": params["bias_scaling_2"],
            "input_scaling": params["input_scaling_2"],
            "ridge_alpha": params["ridge_alpha_2"],
            "noise_val": 0,
        },
        {
            "res_size": params["res_size_3"],
            "sparsity": params["sparsity_3"],
            "alpha": params["alpha_3"],
            "spec_rad": params["spec_rad_3"],
            "bias_scaling": params["bias_scaling_3"],
            "input_scaling": params["input_scaling_3"],
            "ridge_alpha": params["ridge_alpha_3"],
            "noise_val": 0.0,
        },
    ]
    layer_index = 2
    Y_pred, Y_test = deep_henon(
        layer_configs, steps, transient_steps_reservoir, henon_scaled, layer_index
    )

    rmse = root_mean_squared_error(Y_test, Y_pred)
    data.append((Y_test, Y_pred))

 Processing Trial 0/1...

In [76]:
for Y_test, Y_pred in data:
    print(
        f"RMSE: {root_mean_squared_error(Y_test, Y_pred)}, R²: {r2_score(Y_test, Y_pred)}"
    )
generate_henon_grid(data, cols=1).show()

RMSE: 0.06591284108767236, R²: 0.9845222435955091


In [78]:
study.best_params

{'res_size_1': 104,
 'sparsity_1': 0.3225859882243848,
 'alpha_1': 0.7073059104200696,
 'spec_rad_1': 0.5140538455309989,
 'bias_scaling_1': 1.0126952747245566,
 'input_scaling_1': 1.9171107528694222,
 'ridge_alpha_1': 9.862136728961959e-09,
 'noise_val_1': 0.006174945344515407,
 'res_size_2': 67,
 'sparsity_2': 0.03170507087581417,
 'alpha_2': 0.5868004711316878,
 'spec_rad_2': 1.2795871756778885,
 'bias_scaling_2': 0.019949516138543233,
 'input_scaling_2': 1.4925967029194256,
 'ridge_alpha_2': 4.154315604470827e-06,
 'noise_val_2': 0.00015224452244502477,
 'res_size_3': 284,
 'sparsity_3': 0.49365740209372433,
 'alpha_3': 0.7621142735688173,
 'spec_rad_3': 0.2078808501162969,
 'bias_scaling_3': 0.31640568918525547,
 'input_scaling_3': 0.11534289801760045,
 'ridge_alpha_3': 2.094593931997191e-07}

In [77]:
optuna.visualization.plot_param_importances(study).show()

# N Layer Bayesian

In [50]:
num_layers = 5

In [51]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    layer_configs = []
    for i in range(num_layers):
        layer_configs.append(
            {
                "res_size": trial.suggest_int(f"res_size_{i}", 50, 150),
                "sparsity": trial.suggest_float(f"sparsity_{i}", 0.01, 0.5),
                "alpha": trial.suggest_float(f"alpha_{i}", 0.1, 0.8),
                "spec_rad": trial.suggest_float(f"spec_rad_{i}", 0.2, 2.0),
                "bias_scaling": trial.suggest_float(f"bias_scaling_{i}", 1e-2, 2.0),
                "input_scaling": trial.suggest_float(f"input_scaling_{i}", 1e-2, 2.0),
                "ridge_alpha": trial.suggest_float(
                    f"ridge_alpha_{i}", 1e-13, 1e-3, log=True
                ),
                "noise_val": 0.0,
            }
        )

    Y_pred, Y_test = deep_henon(
        layer_configs, steps, transient_steps_reservoir, henon_scaled, num_layers - 1
    )
    return root_mean_squared_error(Y_test, Y_pred)

study = optuna.create_study(directions=["minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=100, n_jobs=-1)

[Optuna] Processing Trial #99...

In [ ]:
def extract_configs(params, num_layers):
    configs = []
    for i in range(num_layers):
        configs.append(
            {
                key: params[f"{key}_{i}"]
                for key in [
                    "res_size",
                    "sparsity",
                    "alpha",
                    "spec_rad",
                    "bias_scaling",
                    "input_scaling",
                    "ridge_alpha",
                ]
            }
        )
        configs[-1]["noise_val"] = 0.0
    return configs

In [ ]:
data = []
for i, trial in enumerate(study.best_trials):
    print(f"\r Processing Trial {i}/{len(study.best_trials)}...", end="", flush=True)
    params = trial.params
    layer_configs = extract_configs(params, num_layers)
    layer_index = -1
    Y_pred, Y_test = deep_henon(
        layer_configs, steps, transient_steps_reservoir, henon_scaled, layer_index
    )

    rmse = root_mean_squared_error(Y_test, Y_pred)
    data.append((Y_test, Y_pred))

 Processing Trial 0/1...

In [ ]:
for Y_test, Y_pred in data:
    print(
        f"RMSE: {root_mean_squared_error(Y_test, Y_pred)}, R²: {r2_score(Y_test, Y_pred)}"
    )
generate_henon_grid(data, cols=1).show()

RMSE: 0.3695316522399219, R²: 0.4165276586217673
